## <u>Exp1</u>(a): Applying Zhu et al.'s model to various benchmark datasets on our system and comparing the results with the ones in the paper

In this experiment, we evaluate the performance of the divide-and-conquer framework proposed in *"Revisiting Single Image Reflection Removal In the Wild"* across multiple real-world benchmark datasets. 

The framework consists of two primary networks:
1. **Reflection Detection Network (RDNet)**: Learns to explicitly characterize the location of local reflections using the Maximum Reflection Filter (MaxRF).
2. **Reflection Removal Network (RRNet)**: Utilizes the estimated location maps to guide the restoration of the clean transmission layer.

### Mathematical Formulation of Network Losses

The authors utilize a supervised learning approach to estimate the reflection location map $\hat{M}_{local}$. The loss function for **RDNet** is defined as:
$$\mathcal{L}_{DNet} = ||M_{local} - \hat{M}_{local}||_{1} + \gamma_{1} * TVLoss(\hat{M}_{local})$$
Where $\gamma_{1}$ represents the balancing weight for the Total Variation (TV) loss, adopted to smooth the estimated results and mitigate artifacts.

For the **RRNet**, the objective function combines content loss and perceptual loss:
$$\mathcal{L}_{RNet} = ||T - \hat{T}||_{1} + \gamma_{2} * ||VGG(T) - VGG(\hat{T})||_{1}$$
Where $VGG(\cdot)$ indicates hierarchical features extracted by the VGG19 network to ensure the restored image maintains high-level structural semantics.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
from analysis_modules.evaluator import ReflectionEvaluator

# Server paths
base_path = '/storage/hdd1/data/amanr.mds2024/robustsirr_test_dataset/'
output_path = './results/all_datasets_eval/'

datasets_to_run = {
    'nature20': 'nature20',
    'real20': 'real20',
    'OpenRR100': 'OpenRR100',
    'SIR2_Postcard': 'SIR2/PostcardDataset',
    'SIR2_SolidObject': 'SIR2/SolidObjectDataset',
    'SIR2_WildScene': 'SIR2/WildSceneDataset'
}

Reflection Detection:  path: ['/home/amanr.mds2024/Developer/cv_proj1/Reflection_RemoVal_CVPR2024', '/home/amanr.mds2024/miniconda3/envs/cv_proj/lib/python311.zip', '/home/amanr.mds2024/miniconda3/envs/cv_proj/lib/python3.11', '/home/amanr.mds2024/miniconda3/envs/cv_proj/lib/python3.11/lib-dynload', '', '/home/amanr.mds2024/miniconda3/envs/cv_proj/lib/python3.11/site-packages', '/home/amanr.mds2024/Developer/cv_proj1', '/ghome/zhuyr/Deref_RW/networks/', '/ghome/zhuyr/Deref_RW/']


In [2]:
# 2. Initialize the Evaluator
evaluator = ReflectionEvaluator(repo_dir='/home/amanr.mds2024/Developer/cv_proj1/Reflection_RemoVal_CVPR2024')

GPU 0 Free Memory: 10.40 GB
GPU 1 Free Memory: 10.41 GB
Models assigned to: cuda:1
Total number of paramerters in EfficientNet networks is 12233232 
Total number of requires_grad paramerters in EfficientNet networks is 12233232 


In [3]:
# 3. Run the loop and collect metrics
results_log = []

for name, rel_path in datasets_to_run.items():
    in_dir = f"{base_path}{rel_path}/blended"
    gt_dir = f"{base_path}{rel_path}/transmission_layer"
    
    psnr, ssim = evaluator.evaluate_dataset(
        dataset_name=name, 
        input_dir=in_dir, 
        gt_dir=gt_dir, 
        output_base_dir=output_path
    )
    
    results_log.append({'Dataset': name, 'PSNR': round(psnr, 4), 'SSIM': round(ssim, 4)})


Evaluating [nature20]: 20 images found.


nature20: 100%|██████████| 20/20 [00:06<00:00,  3.10it/s]



Evaluating [real20]: 20 images found.


real20: 100%|██████████| 20/20 [00:21<00:00,  1.09s/it]



Evaluating [OpenRR100]: 100 images found.


...

SIR2_Postcard: 100%|██████████| 179/179 [00:51<00:00,  3.46it/s]



Evaluating [SIR2_SolidObject]: 200 images found.


SIR2_SolidObject: 100%|██████████| 200/200 [00:50<00:00,  3.93it/s]



Evaluating [SIR2_WildScene]: 101 images found.


SIR2_WildScene: 100%|██████████| 101/101 [00:24<00:00,  4.06it/s]


### Evaluation Metrics: PSNR and SSIM

To evaluate the restoration quality, we utilize **Peak Signal-to-Noise Ratio (PSNR)** and **Structural Similarity Index Measure (SSIM)**. 
* While PSNR calculates pixel-wise mean squared errors, it often falls short in representing human visual perception. 
* SSIM addresses this by analyzing texture, luminance, and contrast, acting as a highly indicative measure of perceived similarity.

Below is the aggregated performance of the model across the benchmarks:

In [5]:
# 4. Display the final summary table
df_results = pd.DataFrame(results_log)
print("\n=== FINAL BENCHMARK RESULTS ===")
display(df_results.T)


=== FINAL BENCHMARK RESULTS ===


,0,1,2,3,4,5
Dataset,nature20,real20,OpenRR100,SIR2_Postcard,SIR2_SolidObject,SIR2_WildScene
PSNR,26.0977,23.7379,25.6488,24.0898,26.6896,27.0049
SSIM,0.8437,0.8127,0.9333,0.8925,0.9262,0.9277


![Benchmark Table](/home/amanr.mds2024/Developer/cv_proj1/Images/Comparison_Table.png)

---

### A few technical details:

We are using the following GPUs whose info are given as follows: `NVIDIA GeForce GTX 1080Ti`.

In [8]:
%%bash
nvidia-smi

Sat Apr 18 15:36:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.03              Driver Version: 575.64.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:05:00.0  On |                  N/A |
| 35%   34C    P8             15W /  260W |     334MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---

Note that in the ImageProcessing class, we have an image processing method, which downsamples the input image if it is bigger than a given user defined threshold T (=1200 for us) pxs in either h or w.  
Here is a formulation of what the method does:

Let the initial RGB input image be represented as a 3D tensor $I \in \mathbb{R}^{H \times W \times 3}$, where $H$ is the height and $W$ is the width. Let the user-defined threshold be $T$ (where $T = 1200$).

### 1. Conditional Downsampling

First, we calculate the scaling factor $s$. The image is only scaled down if the maximum dimension exceeds $T$, which can be expressed as:

$$s = \min\left(1, \frac{T}{\max(H, W)}\right)$$

The intermediate dimensions $H'$ and $W'$ are calculated by scaling $H$ and $W$ and truncating to the nearest integer:

$$H' = \lfloor s \cdot H \rfloor$$
$$W' = \lfloor s \cdot W \rfloor$$

The intermediate resized image $I'$ is obtained using bicubic interpolation:

$$I' = \text{Resize}_{\text{cubic}}(I, (H', W')) \in \mathbb{R}^{H' \times W' \times 3}$$

### 2. Modulo-16 Dimension Adjustment

To ensure the dimensions are compatible with block-based neural network architectures (which typically require dimensions to be divisible by 16), the dimensions are rounded down to the nearest multiple of 16. The new dimensions $H''$ and $W''$ are:

$$H'' = 16 \left\lfloor \frac{H'}{16} \right\rfloor$$
$$W'' = 16 \left\lfloor \frac{W'}{16} \right\rfloor$$

The image is resized a second time to these new dimensions (using OpenCV's default bilinear interpolation):

$$I'' = \text{Resize}_{\text{linear}}(I', (H'', W'')) \in \mathbb{R}^{H'' \times W'' \times 3}$$

*(Note: In the code, `img_resized` represents $I''$.)*

### 3. Tensor Transformation and Normalization

Finally, the image array is converted into a PyTorch tensor. The color channel dimension is permuted from $(H, W, C)$ to $(C, H, W)$, the pixel values are normalized to the range $[0, 1]$, and a batch dimension is added at the 0th index:

$$I_{\text{tensor}} = \frac{\text{permute}(I'', (3, 1, 2))}{255.0}$$

The final output is the tensor $I_{\text{final}}$ ready for network consumption:

$$I_{\text{final}} = \text{unsqueeze}(I_{\text{tensor}}, 0) \in [0, 1]^{1 \times 3 \times H'' \times W''}$$


### A consolidated list of various Interpolation techniques taken from openCV official documentation.
![Interpolation Techniques](/home/amanr.mds2024/Developer/cv_proj1/Images/openCV_Interpolation.png)

---

## <u>Exp1</u>(b): Changing the interpolation technique and comparing the results

#### Using `INTER_AREA` as the interpolation technique

In [6]:
output_path = './results1'

In [7]:
datasets_to_run = {
    'nature20': 'nature20',
    'real20': 'real20',
    'OpenRR100': 'OpenRR100',
    'SIR2_Postcard': 'SIR2/PostcardDataset',
    'SIR2_SolidObject': 'SIR2/SolidObjectDataset',
    'SIR2_WildScene': 'SIR2/WildSceneDataset'
}

In [8]:
evaluator1 = ReflectionEvaluator(repo_dir='/home/amanr.mds2024/Developer/cv_proj1/Reflection_RemoVal_CVPR2024')

GPU 0 Free Memory: 10.40 GB
GPU 1 Free Memory: 10.25 GB
Models assigned to: cuda:0
Total number of paramerters in EfficientNet networks is 12233232 
Total number of requires_grad paramerters in EfficientNet networks is 12233232 


In [9]:
# 3. Run the loop and collect metrics
results_log1 = []

for name, rel_path in datasets_to_run.items():
    in_dir = f"{base_path}{rel_path}/blended"
    gt_dir = f"{base_path}{rel_path}/transmission_layer"
    
    psnr, ssim = evaluator1.evaluate_dataset(
        dataset_name=name, 
        input_dir=in_dir, 
        gt_dir=gt_dir, 
        output_base_dir=output_path
    )
    
    results_log1.append({'Dataset': name, 'PSNR': round(psnr, 4), 'SSIM': round(ssim, 4)})


Evaluating [nature20]: 20 images found.


nature20: 100%|██████████| 20/20 [00:06<00:00,  3.29it/s]



Evaluating [real20]: 20 images found.


real20: 100%|██████████| 20/20 [00:22<00:00,  1.12s/it]



Evaluating [OpenRR100]: 100 images found.


...

SIR2_Postcard: 100%|██████████| 179/179 [00:51<00:00,  3.45it/s]



Evaluating [SIR2_SolidObject]: 200 images found.


SIR2_SolidObject: 100%|██████████| 200/200 [00:52<00:00,  3.79it/s]



Evaluating [SIR2_WildScene]: 101 images found.


SIR2_WildScene: 100%|██████████| 101/101 [00:25<00:00,  3.97it/s]


In [10]:
# 4. Display the final summary table
df_results1 = pd.DataFrame(results_log1)
print("\n=== FINAL BENCHMARK RESULTS ===")
display(df_results1.T)


=== FINAL BENCHMARK RESULTS ===


,0,1,2,3,4,5
Dataset,nature20,real20,OpenRR100,SIR2_Postcard,SIR2_SolidObject,SIR2_WildScene
PSNR,26.0977,23.7379,25.6488,24.0898,26.6896,27.0049
SSIM,0.8437,0.8127,0.9333,0.8925,0.9262,0.9277


---
### Comparison between the two experiment results:

In [11]:
import pandas as pd
from IPython.display import display

# Assuming df_results and df1_results are your original DataFrames
# If 'Dataset' is currently a row (because of a transpose), make sure you are working with the un-transposed data first.
# Here is how to structure it assuming columns are ['Dataset', 'PSNR', 'SSIM']

# 1. Set 'Dataset' as the index for both DataFrames so we can do math on the numeric columns
df1 = df_results.set_index('Dataset')
df2 = df_results1.set_index('Dataset')

# 2. Calculate the difference (e.g., df2 - df1)
# Positive values will mean df1_results performed better; negative means df_results performed better.
df_diff = df2 - df1

# 3. Concatenate them side-by-side using keys to create a MultiIndex column header
df_combined = pd.concat(
    [df1, df2, df_diff], 
    axis=1, 
    keys=['Exp1(a)-Cubic', 'Exp1(b)-Area', 'Difference']
)

# 4. Display the formatted final summary
print("\n=== BENCHMARK COMPARISON & DIFFERENCES ===")
display(df_combined.T) # Transpose if you prefer the metrics as rows and datasets as columns


=== BENCHMARK COMPARISON & DIFFERENCES ===


Dataset             nature20   real20  OpenRR100  SIR2_Postcard  \
Exp1(a)-Cubic PSNR   26.0977  23.7379    25.6488        24.0898   
              SSIM    0.8437   0.8127     0.9333         0.8925   
Exp1(b)-Area  PSNR   26.0977  23.7379    25.6488        24.0898   
              SSIM    0.8437   0.8127     0.9333         0.8925   
Difference    PSNR    0.0000   0.0000     0.0000         0.0000   
              SSIM    0.0000   0.0000     0.0000         0.0000   

Dataset             SIR2_SolidObject  SIR2_WildScene  
Exp1(a)-Cubic PSNR           26.6896         27.0049  
              SSIM            0.9262          0.9277  
Exp1(b)-Area  PSNR           26.6896         27.0049  
              SSIM            0.9262          0.9277  
Difference    PSNR            0.0000          0.0000  
              SSIM            0.0000          0.0000